# ARC-05: Orbital State

## Purpose
Represent a spacecraft translational state in an Earth-Centered Inertial (ECI) frame.

## State representation
- Position: three components in metres (m).
- Velocity: three components in metres per second (m/s).
- State vector: six components ordered as [x, y, z, vx, vy, vz].
- Epoch/time: seconds (s), relative to the simulation start.

## Scientific conventions
- Coordinate frame: ECI.
- Numeric representation: float64.
- State dimension: six.
- Reject non-finite values.
- Do not silently change units or coordinate frames.

## Planned operations
1. Create a validated orbital state.
2. Extract position and velocity.
3. Convert to and from a six-element state vector.
4. Validate state dimensions and finite values.

## Scope
Translational two-body simulation only. Attitude, mass evolution, thrust, and orbital perturbations are outside this initial state model.

In [1]:

import math
import numpy as np

from pydantic import BaseModel, ConfigDict, Field


class OrbitalState(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        frozen=True,
        strict=True,
        allow_inf_nan=False
    )

    position_m: tuple[float, float, float]
    velocity_mps: tuple[float, float, float]
    time_s: float = Field(ge=0)

    def to_array(self):
        return np.asarray(
            (*self.position_m, *self.velocity_mps),
            dtype=np.float64
        )

    @classmethod
    def from_array(cls, state_vector, time_s):
        if isinstance(state_vector, (str, bytes, bytearray, bool, np.bool_)):
            raise ValueError("State vector must be a numeric array")

        try:
            components = list(state_vector)
        except (TypeError, ValueError, OverflowError) as exc:
            raise ValueError("State vector must be a numeric array") from exc
        for component in components:
            if isinstance(component, (bool, np.bool_, str, bytes, bytearray, complex, np.complexfloating)):
                raise ValueError("State vector must contain only real numeric values")
            if isinstance(component, (list, tuple, np.ndarray)):
                raise ValueError("State vector must be one-dimensional")

        try:
            untyped_state = np.asarray(state_vector)
        except (TypeError, ValueError, OverflowError) as exc:
            raise ValueError("State vector must be a numeric array") from exc

        if untyped_state.dtype.kind not in "iuf":
            raise ValueError("State vector must contain only real numeric values")

        try:
            state = np.asarray(untyped_state, dtype=np.float64)
        except (TypeError, ValueError, OverflowError) as exc:
            raise ValueError("State vector must be float64-compatible") from exc

        if state.shape != (6,):
            raise ValueError("State vector must have exactly six components")

        if not np.all(np.isfinite(state)):
            raise ValueError("State vector must contain only finite values")

        if isinstance(time_s, (bool, np.bool_, str, bytes, bytearray)):
            raise ValueError("Time must be a finite numeric scalar")

        try:
            time_value = float(time_s)
        except (TypeError, ValueError, OverflowError) as exc:
            raise ValueError("Time must be a finite numeric scalar") from exc

        if not math.isfinite(time_value) or time_value < 0:
            raise ValueError("Time must be finite and non-negative")

        return cls(
            position_m=tuple(float(x) for x in state[:3]),
            velocity_mps=tuple(float(x) for x in state[3:]),
            time_s=time_value
        )


example_state = OrbitalState(
    position_m=(7_000_000.0, 0.0, 0.0),
    velocity_mps=(0.0, 7_500.0, 0.0),
    time_s=0.0
)

print("Orbital state created successfully")
print("State vector:", example_state.to_array())


Orbital state created successfully
State vector: [7000000.       0.       0.       0.    7500.       0.]


In [2]:

from pydantic import ValidationError

vector = example_state.to_array()
restored = OrbitalState.from_array(vector, example_state.time_s)

assert np.array_equal(restored.to_array(), vector)
assert restored.time_s == example_state.time_s

for invalid in ([1, 2, 3, 4, 5], [1, 2, 3, 4, 5, 6, 7]):
    try:
        OrbitalState.from_array(invalid, 0.0)
        raise AssertionError("Invalid dimension accepted")
    except ValueError:
        pass

for invalid in (
    [1, 2, 3, 4, 5, float("nan")],
    [1, 2, 3, 4, 5, float("inf")]
):
    try:
        OrbitalState.from_array(invalid, 0.0)
        raise AssertionError("Non-finite state accepted")
    except ValueError:
        pass

for invalid in (
    ["1", 2, 3, 4, 5, 6],
    [True, 2, 3, 4, 5, 6],
    [1 + 0j, 2, 3, 4, 5, 6],
):
    try:
        OrbitalState.from_array(invalid, 0.0)
        raise AssertionError("Unexpected state input type accepted")
    except ValueError:
        pass

for invalid_time in (-1.0, float("nan"), float("inf"), True, "0"):
    try:
        OrbitalState.from_array(vector, invalid_time)
        raise AssertionError("Invalid time accepted")
    except ValueError:
        pass

try:
    example_state.time_s = 10.0
    raise AssertionError("Frozen state was modified")
except ValidationError:
    pass

print("Round-trip conversion: PASSED")
print("Invalid dimensions: REJECTED")
print("NaN and Infinity: REJECTED")
print("Invalid time: REJECTED")
print("Frozen state: VERIFIED")


Round-trip conversion: PASSED
Invalid dimensions: REJECTED
NaN and Infinity: REJECTED
Invalid time: REJECTED
Frozen state: VERIFIED
